<a href="https://colab.research.google.com/github/sungyup-jung/projects/blob/main/Non_Modellable_Risk_Factors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **FRTB Non-Modellable Risk Factors (NMRF) Capital & Extreme Value Stress Engine**

Under the Basel Committee's Fundamental Review of the Trading Book (FRTB), every risk factor must pass the Real Price Observation (RPO) test (at least 24 real price observations per year, with no gap exceeding 180 days).

Risk factors failing this test are classified as **Non-Modellable Risk Factors (NMRF)** and cannot be included in the Internal Modal Approach (IMA) Expected Shortfall engine. Instead, they must be capitalized using a standalone **Stressed Scenario* Risk Measure (SSRM)** based on Extreme Value Theory (EVT) and regulatory correlation aggregation rules.

### **Quantitative Methodology**

1. **Observation Gap Analysis**: Scans tick/trade repository timestamps to evaluate rolling 365-day gaps.

2. **Extreme Value Theory (EVT) Tail Scaling**: For unmodellable risk factors with sparse observations, fits a Generalized Pareto Distribution (GPD) over threshold $u$:

$$G_{\xi, \beta}(y) = 1 - \left(1 + \frac{\xi y}{\beta} \right)^{\frac{-1}{\xi}}$$

3. **Regulatory Stress Scenario Risk Measure (SSRM) Aggregation**:

$$SSRM = \sqrt{\sum_{i} I(i) \cdot S^{2}_{i} + \sum_{i}\sum_{j \ne i} \rho_{i,j}S_{i}S_{j}}$$

where $S_{i}$ is the stressed loss for NMRF $i$, and $\rho_{ij}$ is constrained by regulatory directionality buckets.

In [3]:
import numpy as np
import pandas as pd
from scipy.stats import genpareto
from typing import Dict, List, Tuple

class FRTBNMRFEngine:
  """
  FRTB Non-Modellable Risk Factor (NMRF) Identification & Stress Capital Engine.
  Implements Real Price Observation (RPO) gap checks, EVT tail fitting, and
  regulatory SSRM correlation aggregation.
  """
  def __init__(self, observation_dates: Dict[str, List[pd.Timestamp]]):
    self.obs_dates = observation_dates

  def evaluate_rpo_eligibility(self, evaluation_date: pd.Timestamp) -> Dict[str, bool]:
    """
    FRTB RPO Test: >= 24 real observations in trailing 12 months
    AND max observation gap <= 180 days.
    """
    results = {}
    one_year_ago = evaluation_date - pd.DateOffset(years=1)

    for factor, dates in self.obs_dates.items():
      # Filter dates within trailing year
      valid_dates = sorted([d for d in dates if one_year_ago <= d <= evaluation_date])

      if len(valid_dates) < 24:
        results[factor] = False
        continue

      # Check maximum gap between consecutive price observations
      gaps = [(valid_dates[i] - valid_dates[i-1]).days for i in range(1, len(valid_dates))]
      max_gap = max(gaps) if gaps else 365

      results[factor] = (max_gap <= 180)

    return results

  @staticmethod
  def calculate_evt_stress_loss(returns: np.ndarray, quantile: float = 0.999) -> float:
    """
    Fits Generalized Pareto Distribution (GPD) on tail losses (Peaks-Over-Threshold)
    to project extreme stress scenario for unmodellable factors.
    """
    losses = -returns[returns < 0]
    if len(losses) < 10:
      return float(np.max(np.abs(returns)) * 2.5) # Conservative fallback

    threshold = np.quantile(losses, 0.85)
    excesses = losses[losses > threshold] - threshold

    c, loc, scale = genpareto.fit(excesses, floc=0)

    # Extrapolate to regulatory 99.9% stress horizon
    n = len(losses)
    n_u = len(excesses)
    p = 1.0 - quantile

    stressed_loss = threshold + (scale / c) * (((n / n_u) * p) ** (-c) - 1)
    return float(stressed_loss)

  @staticmethod
  def aggregate_ssrm(stress_losses: Dict[str, float], correlation_matrix: np.ndarray) -> float:
    """
    Aggregate individual NMRF stress charges using prescribed Basel correlation formula.
    """
    factors = list(stress_losses.keys())
    s_vec = np.array([stress_losses[f] for f in factors])

    # Quadratic form: sqrt( S^T * Rho * S)
    total_ssrm_sq = np.dot(s_vec, np.dot(correlation_matrix, s_vec))
    return float(np.sqrt(np.maximum(total_ssrm_sq, 0.0)))

if __name__ == "__main__":
  eval_dt = pd.Timestamp("2026-08-01")

  # Generate mock trade observations
  obs_data = {
      "Factor_Liquid_Equity": pd.date_range("2025-08-01", "2026-08-01", freq="W").tolist(),
      "Factor_Exotic_Illiquid_Credit": [pd.Timestamp("2025-09-01"), pd.Timestamp("2026-01-15"), pd.Timestamp("2026-07-10")]
  }

  engine = FRTBNMRFEngine(obs_data)
  rpo_status = engine.evaluate_rpo_eligibility(eval_dt)

  print("=== FRTB RPO Eligibility Assessment ===")
  for factor, status in rpo_status.items():
    print(f"[{'MODELLABLE' if status else 'NMRF / UNMODELLABLE'}] -> {factor}")

  # Stress loss calculation for unmodellable factor
  np.random.seed(42)
  illiquid_returns = np.random.standard_t(df=3, size=200) * 0.03
  evt_loss = engine.calculate_evt_stress_loss(illiquid_returns, quantile=0.999)
  print(f"\nEVT Stressed Scenario Loss (99.9%): {evt_loss*100:.2f}%")

=== FRTB RPO Eligibility Assessment ===
[MODELLABLE] -> Factor_Liquid_Equity
[NMRF / UNMODELLABLE] -> Factor_Exotic_Illiquid_Credit

EVT Stressed Scenario Loss (99.9%): 12.04%
